# Experiment 9: Lid-Driven Cavity Flow

**Aim:** Solve the incompressible cavity problem using the vorticity-streamfunction formulation and analyse the primary recirculation.

**Numerical method:** Explicit vorticity transport, SOR solution of the streamfunction Poisson equation, and wall-vorticity boundary conditions.

> **Scope:** Grid-independence and comparison with benchmark centreline data remain required student validation tasks.

This Google Colab notebook is self-contained. It reconstructs the tested solver inside the temporary Colab runtime, runs it, displays quantitative plots and an inline field contour, and packages every output file for download.

> **Colab workflow:** Choose a run mode, use **Runtime > Run all**, then download the results ZIP from the final cell. `quick` is only a workflow check; use `lab` or an approved `custom` case for submission.

In [ ]:
#@title 1. Prepare the self-contained Colab runtime

from pathlib import Path
import shutil
import subprocess
import sys
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None


def run_solver(script_path, arguments, output_dir, timeout=1800):
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(script_path), *[str(value) for value in arguments]]
    print("Running:", " ".join(command))
    completed = subprocess.run(command, cwd=output_dir, capture_output=True, text=True, timeout=timeout)
    (output_dir / "solver_stdout.log").write_text(completed.stdout, encoding="utf-8")
    (output_dir / "solver_stderr.log").write_text(completed.stderr, encoding="utf-8")
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or f"Solver exited with code {completed.returncode}")
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return output_dir


def read_csv(path):
    data = np.genfromtxt(path, delimiter=",", names=True, dtype=None, encoding="utf-8")
    return np.atleast_1d(data)


def show_summary(path):
    text = Path(path).read_text(encoding="utf-8").strip()
    display(Markdown(f"### Solver summary\n```text\n{text}\n```"))
    return text


def list_outputs(output_dir):
    rows = []
    for path in sorted(Path(output_dir).iterdir()):
        if path.is_file():
            size = path.stat().st_size
            if size < 1024:
                size_text = f"{size} B"
            elif size < 1024**2:
                size_text = f"{size / 1024:.1f} KB"
            else:
                size_text = f"{size / 1024**2:.1f} MB"
            rows.append((path.name, size_text))
    table = "\n".join(f"| `{name}` | {size} |" for name, size in rows)
    display(Markdown(f"### Generated files\n| File | Size |\n|---|---:|\n{table}"))
    return rows


def read_structured_vtk(path):
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    dims_index = next(i for i, line in enumerate(lines) if line.startswith("DIMENSIONS "))
    _, nx, ny, nz = lines[dims_index].split()
    nx, ny, nz = int(nx), int(ny), int(nz)
    if nz != 1:
        raise ValueError("Only 2D structured VTK files are supported by this notebook preview")
    points_index = next(i for i, line in enumerate(lines) if line.startswith("POINTS "))
    npoints = int(lines[points_index].split()[1])
    points = np.array([[float(v) for v in lines[points_index + 1 + k].split()[:3]] for k in range(npoints)])
    x = points[:, 0].reshape(ny, nx)
    y = points[:, 1].reshape(ny, nx)
    data_index = next(i for i, line in enumerate(lines) if line.startswith("POINT_DATA ")) + 1
    scalars = {}
    vectors = {}
    i = data_index
    while i < len(lines):
        if lines[i].startswith("SCALARS "):
            name = lines[i].split()[1]
            i += 2
            values = np.array([float(lines[i + k].split()[0]) for k in range(npoints)])
            scalars[name] = values.reshape(ny, nx)
            i += npoints
        elif lines[i].startswith("VECTORS "):
            name = lines[i].split()[1]
            i += 1
            values = np.array([[float(v) for v in lines[i + k].split()[:3]] for k in range(npoints)])
            vectors[name] = (values[:, 0].reshape(ny, nx), values[:, 1].reshape(ny, nx))
            i += npoints
        else:
            i += 1
    return x, y, scalars, vectors


def plot_vtk_scalar(path, preferred_field=None):
    path = Path(path)
    if not path.is_file():
        print(f"VTK preview skipped: {path.name} was not generated")
        return
    x, y, scalars, _ = read_structured_vtk(path)
    if not scalars:
        print("VTK preview skipped: no scalar field found")
        return
    field = preferred_field if preferred_field in scalars else next(iter(scalars))
    plt.figure(figsize=(8, 4.8))
    image = plt.contourf(x, y, scalars[field], levels=30)
    plt.colorbar(image, label=field.replace("_", " "))
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"{field.replace('_', ' ').title()} - inline field preview")
    plt.axis("equal")
    plt.tight_layout()
    plt.show()


def make_results_zip(output_dir, experiment_number):
    output_dir = Path(output_dir)
    archive = output_dir.parent / f"experiment_{experiment_number:02d}_results.zip"
    if archive.exists():
        archive.unlink()
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as handle:
        for path in sorted(output_dir.rglob("*")):
            if path.is_file():
                handle.write(path, path.relative_to(output_dir))
    print(f"Results archive: {archive} ({archive.stat().st_size / 1024:.1f} KB)")
    return archive


EXPERIMENT = 9
ROOT = Path("/content/cfd_lab_exp09") if IN_COLAB else Path.cwd() / "_colab_runtime_exp09"
CODE_DIR = ROOT / "code"
OUTPUT_DIR = ROOT / "outputs"
CODE_DIR.mkdir(parents=True, exist_ok=True)
(CODE_DIR / "common.py").write_text('from __future__ import annotations\n\nimport csv\nimport math\nfrom pathlib import Path\nfrom typing import Iterable, Mapping\n\nimport numpy as np\n\n\ndef trapezoidal_integral(values: np.ndarray, coordinates: np.ndarray) -> float:\n    y = np.asarray(values, dtype=float).ravel()\n    x = np.asarray(coordinates, dtype=float).ravel()\n    if y.size != x.size or y.size < 2:\n        raise ValueError("trapezoidal integration requires equally sized arrays with at least two points")\n    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))\n\n\ndef write_csv(path: str | Path, header: Iterable[str], rows: Iterable[Iterable[object]]) -> None:\n    with Path(path).open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(list(header))\n        writer.writerows(rows)\n\n\ndef mesh_xy(nx: int, ny: int, lx: float = 1.0, ly: float = 1.0, x0: float = 0.0, y0: float = 0.0) -> tuple[np.ndarray, np.ndarray]:\n    x = np.linspace(x0, x0 + lx, nx)\n    y = np.linspace(y0, y0 + ly, ny)\n    return np.meshgrid(x, y)\n\n\ndef write_structured_grid_vtk(\n    path: str | Path,\n    x: np.ndarray,\n    y: np.ndarray,\n    scalars: Mapping[str, np.ndarray] | None = None,\n    vectors: Mapping[str, tuple[np.ndarray, np.ndarray]] | None = None,\n    title: str = "CFD lab output",\n) -> None:\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y, dtype=float)\n    if x.shape != y.shape or x.ndim != 2:\n        raise ValueError("x and y must be equally shaped 2D arrays")\n    ny, nx = x.shape\n    scalar_map = dict(scalars or {})\n    vector_map = dict(vectors or {})\n    for name, values in scalar_map.items():\n        if np.asarray(values).shape != x.shape:\n            raise ValueError(f"scalar {name} has shape {np.asarray(values).shape}, expected {x.shape}")\n    for name, values in vector_map.items():\n        if len(values) != 2 or np.asarray(values[0]).shape != x.shape or np.asarray(values[1]).shape != x.shape:\n            raise ValueError(f"vector {name} has incompatible shape")\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET STRUCTURED_GRID\\n")\n        handle.write(f"DIMENSIONS {nx} {ny} 1\\n")\n        handle.write(f"POINTS {nx * ny} float\\n")\n        for j in range(ny):\n            for i in range(nx):\n                handle.write(f"{x[j, i]:.12e} {y[j, i]:.12e} 0\\n")\n        handle.write(f"POINT_DATA {nx * ny}\\n")\n        for name, values in scalar_map.items():\n            data = np.asarray(values, dtype=float)\n            handle.write(f"SCALARS {name} float 1\\n")\n            handle.write("LOOKUP_TABLE default\\n")\n            for value in data.ravel(order="C"):\n                handle.write(f"{value:.12e}\\n")\n        for name, values in vector_map.items():\n            vx = np.asarray(values[0], dtype=float)\n            vy = np.asarray(values[1], dtype=float)\n            handle.write(f"VECTORS {name} float\\n")\n            for ax, ay in zip(vx.ravel(order="C"), vy.ravel(order="C")):\n                handle.write(f"{ax:.12e} {ay:.12e} 0\\n")\n\n\ndef write_polyline_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, title: str = "polyline", close: bool = False) -> None:\n    x = np.asarray(x, dtype=float).ravel()\n    y = np.asarray(y, dtype=float).ravel()\n    if x.size != y.size or x.size < 2:\n        raise ValueError("polyline requires equally sized x and y arrays")\n    ids = list(range(x.size))\n    if close:\n        ids.append(0)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {x.size} float\\n")\n        for xi, yi in zip(x, y):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES 1 {len(ids) + 1}\\n")\n        handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_multiline_vtk(path: str | Path, lines: list[tuple[np.ndarray, np.ndarray]], title: str = "lines") -> None:\n    points: list[tuple[float, float]] = []\n    connectivity: list[list[int]] = []\n    for x, y in lines:\n        xa = np.asarray(x, dtype=float).ravel()\n        ya = np.asarray(y, dtype=float).ravel()\n        if xa.size != ya.size or xa.size < 2:\n            raise ValueError("each line needs equally sized x and y arrays")\n        ids = list(range(len(points), len(points) + xa.size))\n        points.extend(zip(xa.tolist(), ya.tolist()))\n        connectivity.append(ids)\n    size = sum(len(ids) + 1 for ids in connectivity)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {len(points)} float\\n")\n        for xi, yi in points:\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES {len(connectivity)} {size}\\n")\n        for ids in connectivity:\n            handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_points_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, scalars: Mapping[str, np.ndarray] | None = None, title: str = "points") -> None:\n    xa = np.asarray(x, dtype=float).ravel()\n    ya = np.asarray(y, dtype=float).ravel()\n    if xa.size != ya.size:\n        raise ValueError("point arrays must have equal size")\n    scalar_map = dict(scalars or {})\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {xa.size} float\\n")\n        for xi, yi in zip(xa, ya):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"VERTICES {xa.size} {2 * xa.size}\\n")\n        for i in range(xa.size):\n            handle.write(f"1 {i}\\n")\n        if scalar_map:\n            handle.write(f"POINT_DATA {xa.size}\\n")\n            for name, values in scalar_map.items():\n                data = np.asarray(values, dtype=float).ravel()\n                if data.size != xa.size:\n                    raise ValueError(f"scalar {name} has wrong size")\n                handle.write(f"SCALARS {name} float 1\\n")\n                handle.write("LOOKUP_TABLE default\\n")\n                for value in data:\n                    handle.write(f"{value:.12e}\\n")\n\n\ndef gradient(field: np.ndarray, dx: float, dy: float) -> tuple[np.ndarray, np.ndarray]:\n    gy, gx = np.gradient(field, dy, dx, edge_order=2)\n    return gx, gy\n\n\ndef laplacian(field: np.ndarray, dx: float, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    out[1:-1, 1:-1] = (\n        (field[1:-1, 2:] - 2.0 * field[1:-1, 1:-1] + field[1:-1, :-2]) / (dx * dx)\n        + (field[2:, 1:-1] - 2.0 * field[1:-1, 1:-1] + field[:-2, 1:-1]) / (dy * dy)\n    )\n    return out\n\n\ndef upwind_derivative_x(field: np.ndarray, velocity: np.ndarray, dx: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[1:-1, :-2]) / dx\n    forward = (field[1:-1, 2:] - field[1:-1, 1:-1]) / dx\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef upwind_derivative_y(field: np.ndarray, velocity: np.ndarray, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[:-2, 1:-1]) / dy\n    forward = (field[2:, 1:-1] - field[1:-1, 1:-1]) / dy\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef poisson_jacobi(phi: np.ndarray, rhs: np.ndarray, dx: float, dy: float, iterations: int, boundary: str = "neumann", mask: np.ndarray | None = None) -> np.ndarray:\n    dx2 = dx * dx\n    dy2 = dy * dy\n    denom = 2.0 * (dx2 + dy2)\n    for _ in range(iterations):\n        old = phi.copy()\n        phi[1:-1, 1:-1] = (\n            (old[1:-1, 2:] + old[1:-1, :-2]) * dy2\n            + (old[2:, 1:-1] + old[:-2, 1:-1]) * dx2\n            - rhs[1:-1, 1:-1] * dx2 * dy2\n        ) / denom\n        if mask is not None:\n            phi[mask > 0.5] = 0.0\n        if boundary == "neumann":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = 0.0\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n        elif boundary == "closed":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = phi[:, -2]\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n            phi -= np.mean(phi)\n        else:\n            raise ValueError("unsupported pressure boundary")\n    return phi\n\n\ndef tdma(a: np.ndarray, b: np.ndarray, c: np.ndarray, d: np.ndarray) -> np.ndarray:\n    a = np.asarray(a, dtype=float).copy()\n    b = np.asarray(b, dtype=float).copy()\n    c = np.asarray(c, dtype=float).copy()\n    d = np.asarray(d, dtype=float).copy()\n    n = d.size\n    if n == 0 or a.size != n or b.size != n or c.size != n:\n        raise ValueError("TDMA arrays must have equal nonzero size")\n    for i in range(1, n):\n        if abs(b[i - 1]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        factor = a[i] / b[i - 1]\n        b[i] -= factor * c[i - 1]\n        d[i] -= factor * d[i - 1]\n    x = np.empty(n, dtype=float)\n    if abs(b[-1]) < 1e-30:\n        raise FloatingPointError("zero TDMA pivot")\n    x[-1] = d[-1] / b[-1]\n    for i in range(n - 2, -1, -1):\n        if abs(b[i]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        x[i] = (d[i] - c[i] * x[i + 1]) / b[i]\n    return x\n\n\ndef relative_l2(numerical: np.ndarray, reference: np.ndarray) -> float:\n    a = np.asarray(numerical, dtype=float)\n    b = np.asarray(reference, dtype=float)\n    denominator = float(np.linalg.norm(b.ravel()))\n    numerator = float(np.linalg.norm((a - b).ravel()))\n    return numerator / max(denominator, 1e-30)\n\n\ndef require_finite(*arrays: np.ndarray) -> None:\n    for array in arrays:\n        if not np.all(np.isfinite(array)):\n            raise FloatingPointError("non-finite numerical field detected")\n\n\ndef positive_float(value: str, name: str) -> float:\n    result = float(value)\n    if not math.isfinite(result) or result <= 0.0:\n        raise ValueError(f"{name} must be positive and finite")\n    return result\n\n\ndef positive_int(value: str, name: str, minimum: int = 1) -> int:\n    result = int(value)\n    if result < minimum:\n        raise ValueError(f"{name} must be at least {minimum}")\n    return result\n', encoding="utf-8")
(CODE_DIR / 'exp09_lid_cavity_psi_omega.py').write_text('from __future__ import annotations\n\nimport csv\nimport sys\n\nimport numpy as np\n\nfrom common import laplacian, require_finite, upwind_derivative_x, upwind_derivative_y, write_multiline_vtk, write_points_vtk, write_structured_grid_vtk\n\n\ndef solve_streamfunction(psi: np.ndarray, omega_field: np.ndarray, dx: float, dy: float, iterations: int, sor: float) -> float:\n    dx2 = dx * dx\n    dy2 = dy * dy\n    denominator = 2.0 / dx2 + 2.0 / dy2\n    residual = 0.0\n    for _ in range(iterations):\n        residual = 0.0\n        for parity in (0, 1):\n            j_indices, i_indices = np.indices((psi.shape[0] - 2, psi.shape[1] - 2))\n            active = ((i_indices + j_indices) % 2) == parity\n            candidate = (\n                (psi[1:-1, 2:] + psi[1:-1, :-2]) / dx2\n                + (psi[2:, 1:-1] + psi[:-2, 1:-1]) / dy2\n                + omega_field[1:-1, 1:-1]\n            ) / denominator\n            old = psi[1:-1, 1:-1].copy()\n            updated = np.where(active, (1.0 - sor) * old + sor * candidate, old)\n            psi[1:-1, 1:-1] = updated\n            residual = max(residual, float(np.max(np.abs(updated - old))))\n        psi[0, :] = 0.0\n        psi[-1, :] = 0.0\n        psi[:, 0] = 0.0\n        psi[:, -1] = 0.0\n    return residual\n\n\ndef velocity_from_streamfunction(psi: np.ndarray, dx: float, dy: float, lid_speed: float) -> tuple[np.ndarray, np.ndarray]:\n    u = np.zeros_like(psi)\n    v = np.zeros_like(psi)\n    u[1:-1, 1:-1] = (psi[2:, 1:-1] - psi[:-2, 1:-1]) / (2.0 * dy)\n    v[1:-1, 1:-1] = -(psi[1:-1, 2:] - psi[1:-1, :-2]) / (2.0 * dx)\n    u[-1, :] = lid_speed\n    u[0, :] = 0.0\n    u[:, 0] = 0.0\n    u[:, -1] = 0.0\n    v[0, :] = 0.0\n    v[-1, :] = 0.0\n    v[:, 0] = 0.0\n    v[:, -1] = 0.0\n    return u, v\n\n\ndef set_wall_vorticity(omega_field: np.ndarray, psi: np.ndarray, dx: float, dy: float, lid_speed: float) -> None:\n    omega_field[0, 1:-1] = -2.0 * psi[1, 1:-1] / (dy * dy)\n    omega_field[-1, 1:-1] = -2.0 * psi[-2, 1:-1] / (dy * dy) - 2.0 * lid_speed / dy\n    omega_field[1:-1, 0] = -2.0 * psi[1:-1, 1] / (dx * dx)\n    omega_field[1:-1, -1] = -2.0 * psi[1:-1, -2] / (dx * dx)\n    omega_field[0, 0] = 0.5 * (omega_field[0, 1] + omega_field[1, 0])\n    omega_field[0, -1] = 0.5 * (omega_field[0, -2] + omega_field[1, -1])\n    omega_field[-1, 0] = 0.5 * (omega_field[-1, 1] + omega_field[-2, 0])\n    omega_field[-1, -1] = 0.5 * (omega_field[-1, -2] + omega_field[-2, -1])\n\n\ndef main() -> int:\n    nx = int(sys.argv[1]) if len(sys.argv) > 1 else 65\n    ny = int(sys.argv[2]) if len(sys.argv) > 2 else 65\n    reynolds = float(sys.argv[3]) if len(sys.argv) > 3 else 100.0\n    lid_speed = float(sys.argv[4]) if len(sys.argv) > 4 else 1.0\n    max_steps = int(sys.argv[5]) if len(sys.argv) > 5 else 2000\n    output_every = int(sys.argv[6]) if len(sys.argv) > 6 else 500\n    poisson_iterations = int(sys.argv[7]) if len(sys.argv) > 7 else 60\n    tolerance = float(sys.argv[8]) if len(sys.argv) > 8 else 1e-8\n    cfl = float(sys.argv[9]) if len(sys.argv) > 9 else 0.8\n    sor = float(sys.argv[10]) if len(sys.argv) > 10 else 1.7\n    if nx < 11 or ny < 11 or reynolds <= 0.0 or lid_speed <= 0.0 or max_steps < 1 or output_every < 1 or poisson_iterations < 1 or tolerance <= 0.0 or not 0.0 < sor < 2.0:\n        raise ValueError("usage: python exp09_lid_cavity_psi_omega.py nx ny Re U maxSteps outputEvery poissonIterations tolerance CFL SOR")\n    x1 = np.linspace(0.0, 1.0, nx)\n    y1 = np.linspace(0.0, 1.0, ny)\n    x, y = np.meshgrid(x1, y1)\n    dx = x1[1] - x1[0]\n    dy = y1[1] - y1[0]\n    nu = lid_speed / reynolds\n    dt = min(0.25 * cfl * min(dx, dy) / lid_speed, 0.05 * min(dx * dx, dy * dy) / nu)\n    psi = np.zeros((ny, nx), dtype=float)\n    omega_field = np.zeros_like(psi)\n    u = np.zeros_like(psi)\n    v = np.zeros_like(psi)\n    history: list[list[float]] = []\n    completed_steps = 0\n    for step in range(1, max_steps + 1):\n        poisson_residual = solve_streamfunction(psi, omega_field, dx, dy, poisson_iterations, sor)\n        u, v = velocity_from_streamfunction(psi, dx, dy, lid_speed)\n        set_wall_vorticity(omega_field, psi, dx, dy, lid_speed)\n        old = omega_field.copy()\n        convection = u * upwind_derivative_x(old, u, dx) + v * upwind_derivative_y(old, v, dy)\n        omega_field[1:-1, 1:-1] = old[1:-1, 1:-1] + dt * (-convection[1:-1, 1:-1] + nu * laplacian(old, dx, dy)[1:-1, 1:-1])\n        set_wall_vorticity(omega_field, psi, dx, dy, lid_speed)\n        change = float(np.sqrt(np.mean((omega_field[1:-1, 1:-1] - old[1:-1, 1:-1]) ** 2)))\n        max_speed = float(np.max(np.sqrt(u * u + v * v)))\n        history.append([step, step * dt, change, poisson_residual, max_speed])\n        completed_steps = step\n        require_finite(psi, omega_field, u, v)\n        if step == 1 or step == max_steps or step % max(1, output_every) == 0:\n            print(f"step={step} time={step * dt:.6e} vorticity_change_l2={change:.6e} poisson_residual={poisson_residual:.6e}")\n        if step % output_every == 0:\n            write_structured_grid_vtk(f"cavity_{step:06d}.vtk", x, y, {"streamfunction": psi, "vorticity": omega_field, "speed": np.sqrt(u * u + v * v)}, {"velocity": (u, v)}, "Lid-driven cavity")\n        if change < tolerance and step > 100:\n            break\n    solve_streamfunction(psi, omega_field, dx, dy, poisson_iterations, sor)\n    u, v = velocity_from_streamfunction(psi, dx, dy, lid_speed)\n    write_structured_grid_vtk("cavity_final.vtk", x, y, {"streamfunction": psi, "vorticity": omega_field, "speed": np.sqrt(u * u + v * v)}, {"velocity": (u, v)}, "Lid-driven cavity final")\n    write_multiline_vtk("cavity_boundaries.vtk", [(np.array([0.0, 1.0]), np.array([0.0, 0.0])), (np.array([1.0, 1.0]), np.array([0.0, 1.0])), (np.array([1.0, 0.0]), np.array([1.0, 1.0])), (np.array([0.0, 0.0]), np.array([1.0, 0.0]))], "Cavity boundaries")\n    seed = np.linspace(0.05, 0.95, 15)\n    sx, sy = np.meshgrid(seed, seed)\n    write_points_vtk("cavity_stream_seeds.vtk", sx.ravel(), sy.ravel(), title="Cavity stream seeds")\n    i_mid = int(np.argmin(np.abs(x1 - 0.5)))\n    j_mid = int(np.argmin(np.abs(y1 - 0.5)))\n    with open("cavity_centerline_profiles.csv", "w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["vertical_y", "u_at_x_0p5", "horizontal_x", "v_at_y_0p5"])\n        nrows = max(nx, ny)\n        for k in range(nrows):\n            j = min(int(round(k * (ny - 1) / max(nrows - 1, 1))), ny - 1)\n            i = min(int(round(k * (nx - 1) / max(nrows - 1, 1))), nx - 1)\n            writer.writerow([y1[j], u[j, i_mid], x1[i], v[j_mid, i]])\n    with open("cavity_history.csv", "w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["step", "time", "vorticity_change_l2", "poisson_residual_linf", "maximum_speed"])\n        writer.writerows(history)\n    minimum_index = np.unravel_index(np.argmin(psi), psi.shape)\n    with open("cavity_summary.txt", "w", encoding="utf-8") as handle:\n        handle.write("Lid-driven cavity vorticity-streamfunction solution\\n")\n        handle.write(f"nx {nx}\\nny {ny}\\nRe {reynolds:.12e}\\nlid_speed {lid_speed:.12e}\\nnu {nu:.12e}\\ndt {dt:.12e}\\ncompleted_steps {completed_steps}\\n")\n        handle.write(f"final_vorticity_change_l2 {history[-1][2]:.12e}\\nminimum_streamfunction {float(np.min(psi)):.12e}\\n")\n        handle.write(f"primary_vortex_x {x1[minimum_index[1]]:.12e}\\nprimary_vortex_y {y1[minimum_index[0]]:.12e}\\n")\n        handle.write("benchmark_requirement compare_centerlines_with_Ghia_Ghia_Shin\\n")\n    print("Wrote cavity_final.vtk, cavity_boundaries.vtk, cavity_stream_seeds.vtk, cavity_centerline_profiles.csv, cavity_history.csv, cavity_summary.txt")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
SCRIPT = CODE_DIR / 'exp09_lid_cavity_psi_omega.py'
print("Runtime:", "Google Colab" if IN_COLAB else "local notebook test")
print("Working folder:", ROOT)
print("No package installation or external file upload is required.")


## Parameters

Use `quick` to test the workflow, `lab` for the prescribed case, and `custom` only when changing the form values for the assignment.

In [ ]:
#@title 2. Select run mode and parameters
RUN_MODE = "quick" #@param ["quick", "lab", "custom"]

nx = 65 #@param {type:"integer"}
ny = 65 #@param {type:"integer"}
Re = 100.0 #@param {type:"number"}
U_lid = 1.0 #@param {type:"number"}
max_steps = 2000 #@param {type:"integer"}
output_every = 500 #@param {type:"integer"}
poisson_iterations = 60 #@param {type:"integer"}
tolerance = 1e-08 #@param {type:"number"}
CFL = 0.8 #@param {type:"number"}
SOR = 1.7 #@param {type:"number"}

QUICK_ARGS = [21, 21, 100, 1.0, 10, 10, 10, 1e-05, 0.8, 1.5]
LAB_ARGS = [65, 65, 100, 1.0, 2000, 500, 60, 1e-08, 0.8, 1.7]
CUSTOM_ARGS = [nx, ny, Re, U_lid, max_steps, output_every, poisson_iterations, tolerance, CFL, SOR]
ARGS = QUICK_ARGS if RUN_MODE == "quick" else LAB_ARGS if RUN_MODE == "lab" else CUSTOM_ARGS
print("Run mode:", RUN_MODE)
print("Arguments:", ARGS)


## Run the solver

In [ ]:
#@title 3. Run the numerical solver
OUT = run_solver(SCRIPT, ARGS, OUTPUT_DIR)


## Numerical summary

In [ ]:
#@title 4. Read the solver summary
summary_text = show_summary(OUT / 'cavity_summary.txt')


## Quantitative plots

In [ ]:
#@title 5. Plot the primary numerical results
profiles = read_csv(OUT / "cavity_centerline_profiles.csv")
plt.figure(figsize=(7, 4))
plt.plot(profiles["u_at_x_0p5"], profiles["vertical_y"])
plt.xlabel("u at x = 0.5")
plt.ylabel("y")
plt.title("Vertical centreline velocity")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(profiles["horizontal_x"], profiles["v_at_y_0p5"])
plt.xlabel("x")
plt.ylabel("v at y = 0.5")
plt.title("Horizontal centreline velocity")
plt.grid(True)
plt.show()

history = read_csv(OUT / "cavity_history.csv")
plt.figure(figsize=(7, 4))
plt.semilogy(history["step"], history["vorticity_change_l2"])
plt.xlabel("Step")
plt.ylabel("Vorticity-change L2 norm")
plt.title("Approach to steady state")
plt.grid(True)
plt.show()

## Inline field visualization

This preview removes the need for ParaView during the basic laboratory workflow. The VTK file is still included in the downloaded results for optional advanced post-processing.

In [ ]:
#@title 6. Display the final field and generated files
plot_vtk_scalar(OUT / 'cavity_final.vtk', 'streamfunction')
files = list_outputs(OUT)


## Download results

In [ ]:
#@title 7. Package and download the results
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
RESULTS_ZIP = make_results_zip(OUT, 9)
if DOWNLOAD_RESULTS:
    if IN_COLAB:
        colab_files.download(str(RESULTS_ZIP))
    else:
        print("Local test mode. Archive retained at:", RESULTS_ZIP)
else:
    print("Set DOWNLOAD_RESULTS to True and run this cell when ready to download.")


## 7. Compulsory assignment

Run $Re=100$ and $Re=400$. Compare centreline velocities, main-vortex position, convergence history, and secondary-corner-vortex development.

## Report checklist

- State the governing equations and boundary conditions.
- Report the complete parameter set and grid.
- Include residual or error history.
- Include the required quantitative comparison.
- Include at least one correctly labelled inline field figure; VTK/ParaView figures are optional unless specifically assigned.
- Distinguish observed numerical behaviour from physical interpretation.
- State the limitations of the supplied teaching model.